In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import List, Dict, Tuple

# Add necessary paths
sys.path.append('.')
sys.path.append('./plotting_notebooks')

# Import the loading functions from the analysis script
from plotting_notebooks_protocols.protocol_plotting_utils import (
    load_in_combined_results,
    load_protocol_intervention_transcripts,
    load_protocol_intervention_data
)

def load_data():
    """Load and prepare all data for analysis"""
    print("Loading data...")
    
    # Load combined results
    combined_results = load_in_combined_results(
        include_failed_attempts=True, 
        include_failed_main_task=True
    )
    
    # Load protocol data
    control_protocol_list = [
        'defer_to_instructed_trusted_low_context', 
        'defer_to_trusted', 
        'defer_to_instructed_trusted', 
        'defer_to_trusted_with_feedback', 
        'resample_with_feedback'
    ]
    
    protocol_results_list = []
    intervention_data_list = []
    
    for control_protocol in control_protocol_list:
        try:
            protocol_results = load_protocol_intervention_transcripts(protocol_name=control_protocol)
            intervention_data = load_protocol_intervention_data(protocol_name=control_protocol)
            protocol_results_list.append(protocol_results)
            intervention_data_list.append(intervention_data)
        except Exception as e:
            print(f"Error loading {control_protocol}: {e}")
    
    protocol_results = pd.concat(protocol_results_list)
    intervention_data = pd.concat(intervention_data_list)
    
    # Merge protocol results with intervention data
    protocol_results_and_data = pd.merge(
        protocol_results, 
        intervention_data, 
        on=['task_name', 'transcript_type', 'protocol_name', 'main_agent', 'monitor_agent', 'file_name'], 
        how='inner'
    )
    
    return combined_results, protocol_results, intervention_data, protocol_results_and_data

def plot_protocol_tradeoff(protocol_results_and_data, combined_results):
    """
    Create a scatter plot showing the tradeoff between main task success and side task blocking
    for each protocol at different thresholds
    """
    plt.figure(figsize=(12, 8))
    
    # Filter to include only malicious attempts (with side tasks)
    malicious_data = protocol_results_and_data[
        protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'
    ]
    
    # Group by protocol and threshold
    grouped = malicious_data.groupby(['protocol_name', 'suspicion_threshold'])
    
    # Colors and markers for different protocols
    protocols = malicious_data.protocol_name.unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(protocols)))
    markers = ['o', 's', 'D', '^', 'v', '>', '<', 'p', '*']
    
    # Plot each protocol with threshold labels
    for i, protocol in enumerate(protocols):
        protocol_data = grouped.get_group((protocol, ))
        thresholds = sorted(protocol_data.suspicion_threshold.unique())
        
        x_values = []  # Main task success rate
        y_values = []  # Side task blocking rate (1 - side task success)
        
        for threshold in thresholds:
            data_at_threshold = malicious_data[
                (malicious_data.protocol_name == protocol) & 
                (malicious_data.suspicion_threshold == threshold)
            ]
            
            main_task_success = data_at_threshold.main_task_success.mean()
            side_task_success = data_at_threshold.side_task_evaluation.mean()
            side_task_blocking = 1 - side_task_success
            
            x_values.append(main_task_success)
            y_values.append(side_task_blocking)
            
            # Plot each point
            plt.scatter(
                main_task_success, 
                side_task_blocking, 
                color=colors[i],
                marker=markers[i % len(markers)], 
                s=100, 
                label=f"{protocol}, t={threshold}" if threshold == thresholds[0] else None
            )
            
            # Add threshold labels
            plt.annotate(
                f"{threshold}", 
                (main_task_success, side_task_blocking),
                xytext=(5, 5), 
                textcoords='offset points'
            )
        
        # Connect points for the same protocol
        plt.plot(x_values, y_values, '-', color=colors[i], alpha=0.5)
    
    # Add no-intervention baseline
    baseline_data = combined_results[
        combined_results.transcript_type != 'transcripts_no_secondary_task'
    ]
    baseline_main = baseline_data.main_task_success.mean()
    baseline_side = 1 - baseline_data.side_task_evaluation.mean()
    
    plt.scatter(
        baseline_main, baseline_side, 
        marker='X', color='red', s=200,
        label='No intervention'
    )
    
    plt.annotate(
        "No intervention", 
        (baseline_main, baseline_side),
        xytext=(10, -10), 
        textcoords='offset points'
    )
    
    # Format plot
    plt.xlabel('Main Task Success Rate', fontsize=12)
    plt.ylabel('Side Task Blocking Rate (1 - side task success)', fontsize=12)
    plt.title('Protocol Performance Tradeoff: Main Task Success vs Side Task Blocking', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, 1.05)
    plt.ylim(0, 1.05)
    
    # Add diagonal reference line for equal tradeoff
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    
    # Add "better performance" arrow
    plt.annotate(
        "", xy=(0.95, 0.95), xytext=(0.7, 0.7),
        arrowprops=dict(facecolor='green', shrink=0.05, width=2, headwidth=10),
    )
    plt.text(0.8, 0.8, "Better", color='green', ha='center', va='center', fontsize=12)
    
    plt.legend(loc='lower left', bbox_to_anchor=(0, 0), ncol=2)
    plt.tight_layout()
    plt.show()

def plot_threshold_sensitivity(protocol_results_and_data):
    """
    Create line plots showing how varying threshold affects main task and side task performance
    for each protocol
    """
    plt.figure(figsize=(15, 10))
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14), sharex=True)
    
    # Filter to include only malicious attempts (with side tasks)
    malicious_data = protocol_results_and_data[
        protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'
    ]
    
    # Colors for different protocols
    protocols = malicious_data.protocol_name.unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(protocols)))
    
    # Plot main task success vs threshold
    for i, protocol in enumerate(protocols):
        protocol_data = malicious_data[malicious_data.protocol_name == protocol]
        thresholds = sorted(protocol_data.suspicion_threshold.unique())
        
        main_task_values = []
        side_task_values = []
        
        for threshold in thresholds:
            data_at_threshold = protocol_data[protocol_data.suspicion_threshold == threshold]
            main_task_values.append(data_at_threshold.main_task_success.mean())
            side_task_values.append(1 - data_at_threshold.side_task_evaluation.mean())
        
        # Plot main task success
        ax1.plot(thresholds, main_task_values, 'o-', color=colors[i], linewidth=2, label=protocol)
        
        # Plot side task blocking
        ax2.plot(thresholds, side_task_values, 's-', color=colors[i], linewidth=2, label=protocol)
    
    # Format plots
    ax1.set_ylabel('Main Task Success Rate', fontsize=12)
    ax1.set_title('Effect of Suspicion Threshold on Main Task Success', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1.05)
    ax1.legend(loc='best')
    
    ax2.set_xlabel('Suspicion Threshold', fontsize=12)
    ax2.set_ylabel('Side Task Blocking Rate (1 - side task success)', fontsize=12)
    ax2.set_title('Effect of Suspicion Threshold on Side Task Blocking', fontsize=14)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1.05)
    ax2.legend(loc='best')
    
    plt.tight_layout()
    plt.show()

def plot_protocol_comparison(protocol_results_and_data, combined_results):
    """
    Create a bar chart directly comparing protocols at their optimal thresholds
    """
    # Filter to include only malicious attempts (with side tasks)
    malicious_data = protocol_results_and_data[
        protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'
    ]
    
    # Find optimal threshold for each protocol
    optimal_configs = []
    protocols = malicious_data.protocol_name.unique()
    
    for protocol in protocols:
        protocol_data = malicious_data[malicious_data.protocol_name == protocol]
        thresholds = sorted(protocol_data.suspicion_threshold.unique())
        
        best_f1 = 0
        best_threshold = None
        best_main = 0
        best_side = 0
        
        for threshold in thresholds:
            data_at_threshold = protocol_data[protocol_data.suspicion_threshold == threshold]
            main_success = data_at_threshold.main_task_success.mean()
            side_blocking = 1 - data_at_threshold.side_task_evaluation.mean()
            
            # Calculate F1-like score to balance both objectives
            if (main_success + side_blocking) > 0:
                f1 = 2 * (main_success * side_blocking) / (main_success + side_blocking)
            else:
                f1 = 0
                
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                best_main = main_success
                best_side = side_blocking
        
        optimal_configs.append({
            'protocol': protocol,
            'threshold': best_threshold,
            'main_task_success': best_main,
            'side_task_blocking': best_side,
            'f1_score': best_f1
        })
    
    # Add baseline (no intervention)
    baseline_data = combined_results[
        combined_results.transcript_type != 'transcripts_no_secondary_task'
    ]
    baseline_main = baseline_data.main_task_success.mean()
    baseline_side = 1 - baseline_data.side_task_evaluation.mean()
    
    if (baseline_main + baseline_side) > 0:
        baseline_f1 = 2 * (baseline_main * baseline_side) / (baseline_main + baseline_side)
    else:
        baseline_f1 = 0
        
    optimal_configs.append({
        'protocol': 'No intervention',
        'threshold': 'N/A',
        'main_task_success': baseline_main,
        'side_task_blocking': baseline_side,
        'f1_score': baseline_f1
    })
    
    # Create DataFrame and sort by F1 score
    optimal_df = pd.DataFrame(optimal_configs)
    optimal_df = optimal_df.sort_values('f1_score', ascending=False)
    
    # Plot
    plt.figure(figsize=(14, 8))
    
    # Set up bars
    x = np.arange(len(optimal_df))
    width = 0.35
    
    # Create bars
    bars1 = plt.bar(x - width/2, optimal_df['main_task_success'], width, label='Main Task Success')
    bars2 = plt.bar(x + width/2, optimal_df['side_task_blocking'], width, label='Side Task Blocking')
    
    # Add F1 score as text
    for i, (_, row) in enumerate(optimal_df.iterrows()):
        plt.text(i, 1.05, f"F1: {row['f1_score']:.2f}", ha='center', va='bottom', rotation=0)
        plt.text(i, -0.05, f"t={row['threshold']}", ha='center', va='top')
    
    # Format plot
    plt.xlabel('Protocol')
    plt.ylabel('Rate')
    plt.title('Protocol Comparison at Optimal Thresholds')
    plt.xticks(x, optimal_df['protocol'], rotation=45, ha='right')
    plt.ylim(0, 1.2)
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_task_heatmap(protocol_results_and_data):
    """
    Create a heatmap showing which protocols work best for specific tasks
    """
    # Filter to include only malicious attempts (with side tasks)
    malicious_data = protocol_results_and_data[
        protocol_results_and_data.transcript_type != 'transcripts_no_secondary_task'
    ]
    
    # Calculate F1-like score for each protocol-task-threshold combination
    result_data = []
    for protocol in malicious_data.protocol_name.unique():
        for task in malicious_data.task_name.unique():
            task_protocol_data = malicious_data[
                (malicious_data.protocol_name == protocol) &
                (malicious_data.task_name == task)
            ]
            
            if len(task_protocol_data) == 0:
                continue
                
            # Find best threshold for this task-protocol combination
            best_f1 = 0
            best_threshold = None
            
            for threshold in task_protocol_data.suspicion_threshold.unique():
                data_at_threshold = task_protocol_data[task_protocol_data.suspicion_threshold == threshold]
                
                if len(data_at_threshold) == 0:
                    continue
                    
                main_success = data_at_threshold.main_task_success.mean()
                side_blocking = 1 - data_at_threshold.side_task_evaluation.mean()
                
                # Calculate F1-like score
                if (main_success + side_blocking) > 0:
                    f1 = 2 * (main_success * side_blocking) / (main_success + side_blocking)
                else:
                    f1 = 0
                    
                if f1 > best_f1:
                    best_f1 = f1
                    best_threshold = threshold
            
            if best_threshold is not None:
                result_data.append({
                    'protocol': protocol,
                    'task': task,
                    'f1_score': best_f1,
                    'threshold': best_threshold
                })
    
    # Create DataFrame for heatmap
    result_df = pd.DataFrame(result_data)
    heatmap_data = result_df.pivot(index='task', columns='protocol', values='f1_score')
    threshold_data = result_df.pivot(index='task', columns='protocol', values='threshold')
    
    # Plot
    plt.figure(figsize=(14, 10))
    
    # Create heatmap with protocol-task F1 scores
    ax = sns.heatmap(
        heatmap_data, 
        annot=True, 
        cmap='YlGnBu', 
        fmt='.2f',
        linewidths=0.5,
        vmin=0, 
        vmax=1
    )
    
    # Add threshold values as text
    for i, task in enumerate(heatmap_data.index):
        for j, protocol in enumerate(heatmap_data.columns):
            if not pd.isna(threshold_data.loc[task, protocol]):
                plt.text(
                    j+0.5, i+0.8, 
                    f"t={threshold_data.loc[task, protocol]}", 
                    ha='center', va='center', 
                    color='black', fontsize=8
                )
    
    plt.title('Protocol Performance by Task (F1 Score at Best Threshold)')
    plt.tight_layout()
    plt.show()

def main():
    # Load data
    combined_results, protocol_results, intervention_data, protocol_results_and_data = load_data()
    
    # Generate all plots
    plot_protocol_tradeoff(protocol_results_and_data, combined_results)
    plot_threshold_sensitivity(protocol_results_and_data)
    plot_protocol_comparison(protocol_results_and_data, combined_results)
    plot_task_heatmap(protocol_results_and_data)

if __name__ == "__main__":
    main() 